# Churn Prediction Pipeline Using Temporal Behavior

Dự án này xây dựng mô hình dự báo rời bỏ dịch vụ của khách hàng (Customer Churn Prediction) dựa trên lịch sử tương tác theo thời gian.
Notebook này hoạt động như một cẩm nang (walkthrough) trình bày toàn bộ quy trình từ dữ liệu thô, kỹ nghệ đặc trưng, thực nghiệm so sánh thuật toán, tối ưu hóa đặc trưng, đến huấn luyện, hiệu chuẩn và kiểm định mô hình sản xuất.

**Thành phần chính của Pipeline:**
1. Trích xuất đặc trưng thời gian (Lags, Rolling, Trend, Recency)
2. Gán nhãn Churn tương lai (Prediction Window)
3. Điền khuyết dữ liệu chống rò rỉ (Chronological Imputation)
4. Huấn luyện thử nghiệm so sánh thuật toán (Model Comparison)
5. Lựa chọn đặc trưng tự động (Feature Selection)
6. Huấn luyện & Hiệu chuẩn xác suất (LightGBM + Platt Scaling)
7. Lưu trữ & Kiểm định gói Artifacts


## 1. Bài toán (Problem Definition)

Bài toán đặt ra là dự đoán khả năng khách hàng rời bỏ dịch vụ (`churn_next_30d = 1`) hoặc hạ cấp gói cước và ngừng hoạt động trong vòng **30 ngày tiếp theo** (prediction horizon) tính từ thời điểm quan sát snapshot $S$.
Mô hình cần nhận diện sớm những khách hàng có nguy cơ rời bỏ để doanh nghiệp đưa ra các biện pháp chăm sóc hoặc ưu đãi phù hợp nhằm giữ chân khách hàng.


## 2. Dataset & Silver Layer

Dữ liệu thô của dự án được tổ chức và lưu trữ dưới dạng các tệp sự kiện Parquet phân vùng tại **Silver Layer**:
- **`churn_customers`**: Thông tin nhân khẩu học của khách hàng (giới tính, thành phố, ngày đăng ký, ngày đóng tài khoản).
- **`churn_subscriptions`**: Nhật ký thay đổi gói cước (nâng cấp, hạ cấp gói cước).
- **`churn_product_usage`**: Tần suất hoạt động của khách hàng trên ứng dụng/hệ thống.
- **`churn_orders`**: Nhật ký giao dịch đơn đặt hàng và chi tiêu.
- **`churn_payments`**: Kết quả giao dịch thanh toán hóa đơn.
- **`churn_support_tickets`**: Các ticket phản hồi khiếu nại và điểm số CSAT.


## 3. Snapshot Date, Observation Window và Prediction Window

Để thực hiện mô hình hóa thời gian, chúng ta chia timeline của mỗi khách hàng thành các mốc quan sát hàng tháng (Snapshot Dates).

```text
QUÁ KHỨ (Lịch sử hành vi)                  TƯƠNG LAI (Mục tiêu dự báo)

Observation Window
<----------------------------------------|------------------------>
                                   Snapshot Date (S)
                                         │
                                         │◄─────── 30 ngày ───────►
                                         │   Prediction Window
```

**Nguyên tắc cốt lõi:**
- **Snapshot Date (S)**: Ngày 1 hàng tháng.
- **Quan sát đặc trưng (Features)**: Chỉ tổng hợp và tính toán các đặc trưng (lags, rolling, recency) từ các sự kiện xảy ra **trước hoặc đúng ngày S** ($Event \le S$).
- **Gán nhãn Churn (Labels)**: Chỉ sử dụng dữ liệu hành vi xảy ra trong **30 ngày tiếp theo** của tương lai ($S < Event \le S + 30	ext{ ngày}$). Nếu khách hàng hủy tài khoản hoặc hạ cấp gói cước và không có hoạt động trong khoảng này, gán nhãn `1`, ngược lại gán nhãn `0`.


## 4. Load Data

Chúng ta bắt đầu bằng việc import các thư viện cần thiết và thiết lập cấu hình chung.


In [ ]:
import os
import joblib
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)
import xgboost as xgb
import lightgbm as lgb

from src import config
from src.data.load_silver import load_silver_table

# Cấu hình logging tối giản
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("JupyterNotebook")

print("Imports completed successfully!")


## 5. Xây Temporal Base

Temporal Base là bảng lưới liên kết toàn bộ khách hàng với các tháng quan sát tương ứng từ khi họ đăng ký (`signup_date`) đến khi đóng tài khoản (`closed_date`).
Dữ liệu thô từ các bảng log sự kiện sau đó được tổng hợp theo tháng (monthly aggregates) để gán vào lưới này.


In [ ]:
# Minh họa logic nạp dữ liệu thô từ Silver Layer
customers = load_silver_table("churn_customers")
orders = load_silver_table("churn_orders")
usage = load_silver_table("churn_product_usage")

print(f"Bảng khách hàng: {customers.shape}")
print(f"Bảng đơn hàng: {orders.shape}")
print(f"Bảng tần suất sử dụng: {usage.shape}")


## 6. Lag Features

Đặc trưng Lag dịch chuyển hành vi của khách hàng trong quá khứ ngược lại 1, 2, và 3 tháng để giúp mô hình phát hiện xu hướng giảm sút mức độ gắn kết gần đây.

**Công thức tổng quát:**
$$usage\_lag\_1(t) = usage(t-1)$$
$$usage\_lag\_2(t) = usage(t-2)$$


## 7. Rolling Features

Đặc trưng Rolling tính toán các thống kê tổng hợp (tổng, trung bình, độ lệch chuẩn, tối thiểu, tối đa) trên các cửa sổ trượt 1, 3, và 6 tháng trước ngày snapshot.

**Công thức minh họa:**
$$usage\_rolling\_mean\_3m(t) = \frac{1}{3} \sum_{i=1}^{3} usage(t-i)$$


## 8. Trend Features

Đặc trưng Trend đo lường biến động tương đối hoặc độ dốc xu hướng của khách hàng trong các tháng gần nhất.
- **Biến động phần trăm (Percentage Change)**:
  $$usage\_pct\_change\_1m(t) = \frac{usage(t-1) - usage(t-2)}{usage(t-2)}$$
- **Độ dốc (Slope)**: Hệ số hồi quy tuyến tính khớp trên 3 tháng hoạt động gần nhất.


## 9. Recency Features

Đặc trưng Recency tính khoảng cách (số ngày) kể từ lần tương tác cuối cùng của khách hàng đối với các hành vi như: đơn hàng cuối, thanh toán cuối, hoặc lần hạ cấp gói cước cuối.
Sử dụng phương pháp nối ngược thời gian (`pd.merge_asof` với `allow_exact_matches=False`) để bảo đảm zero lookahead bias.

**Công thức:**
$$days\_since\_last\_order(S) = S - last\_order\_date$$


## 10. Churn Label

Gán nhãn Churn tương lai dựa trên dữ liệu 30 ngày sau snapshot.
Khách hàng được coi là Churn (`churn_next_30d = 1`) nếu có sự kiện đóng tài khoản (`closed_date`) hoặc hạ cấp dịch vụ (`downgrade` gói cước) và không phát sinh bất kỳ tương tác hoạt động nào trong vòng 30 ngày kế tiếp.


## 11. Final Temporal Dataset

Toàn bộ các đặc trưng và nhãn được gộp lại thành tệp dữ liệu duy nhất [`output/churn_temporal_dataset.parquet`](file:///d:/Intern%20Data/output/churn_temporal_dataset.parquet).
Tệp dữ liệu này đã được áp dụng quy trình **Chronological Preprocessing (Median Imputation)** fit trên Train để xử lý các giá trị trống một cách chuẩn hóa trước khi đưa vào mô hình.


In [ ]:
# Nạp tập dữ liệu preprocessed tích hợp đầy đủ
dataset_path = Path("output/churn_temporal_dataset.parquet")
df = pd.read_parquet(dataset_path)
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])

print(f"Kích thước tập dữ liệu Churn Temporal: {df.shape}")
print(f"Khoảng thời gian snapshot: {df['snapshot_date'].min().date()} đến {df['snapshot_date'].max().date()}")
df.head()


## 12. Data Quality & Leakage Audit

Chúng ta thực hiện kiểm tra kiểm duyệt dữ liệu để đảm bảo không có rò rỉ thông tin tương lai và không có giá trị lỗi vô cực trước khi chia dữ liệu.


In [ ]:
# Kiểm định rò rỉ mục tiêu và vô cực
meta_cols = ["customer_id", "snapshot_date", "churn_next_30d"]
feature_cols = [c for c in df.columns if c not in meta_cols]

# 1. Kiểm tra cột rò rỉ tương lai
forbidden_cols = ["closed_date", "signup_date", "birth_date"]
for col in forbidden_cols:
    assert col not in feature_cols, f"Phát hiện cột cấm: {col}"
    
# 2. Kiểm tra cột bắt đầu bằng future_
for col in feature_cols:
    assert not col.startswith("future_"), f"Phát hiện rò rỉ tương lai: {col}"

# 3. Kiểm tra giá trị vô cực
inf_count = np.isinf(df[feature_cols]).sum().sum()
assert inf_count == 0, f"Phát hiện {inf_count} giá trị vô cực (+/-inf)!"

print("PASS: Kiểm duyệt dữ liệu hoàn tất. Dữ liệu sạch, không chứa giá trị vô cực và rò rỉ tương lai.")


## 13. Time-based Split

Để đánh giá chính xác khả năng dự báo của mô hình trong tương lai thực tế, dữ liệu được chia theo trình tự thời gian (Chronological Split):

```text
Quá khứ                                          Tương lai
┌───────────────────────────┬───────────────┬──────────────────┐
│           Train           │  Validation   │    Clean Test    │
│    (Huấn luyện chính)     │ (Hiệu chuẩn)  │ (Đánh giá độc lập)│
└───────────────────────────┴───────────────┴──────────────────┘
                      2025-08-01        2026-02-01       2026-06-01
```

**Mốc thời gian cụ thể:**
- **Train (Tập huấn luyện)**: Các snapshot $\le$ `2025-08-01` (Học luật hành vi lịch sử).
- **Validation (Tập hiệu chuẩn)**: Các snapshot từ `2025-09-01` đến `2026-02-01` (Tối ưu hóa ngưỡng quyết định và Platt Scaling).
- **Test (Tập kiểm thử độc lập)**: Các snapshot từ `2026-03-01` đến `2026-06-01` (Mature Test Cohort).


In [ ]:
# Chia dữ liệu theo trình tự thời gian
train_end = pd.Timestamp("2025-08-01")
val_start = pd.Timestamp("2025-09-01")
val_end = pd.Timestamp("2026-02-01")
test_start = pd.Timestamp("2026-03-01")
test_end = pd.Timestamp("2026-06-01")

train_df = df[df["snapshot_date"] <= train_end].copy()
val_df = df[(df["snapshot_date"] >= val_start) & (df["snapshot_date"] <= val_end)].copy()
test_df = df[(df["snapshot_date"] >= test_start) & (df["snapshot_date"] <= test_end)].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df["churn_next_30d"].to_numpy()

X_val = val_df[feature_cols].copy()
y_val = val_df["churn_next_30d"].to_numpy()

X_test = test_df[feature_cols].copy()
y_test = test_df["churn_next_30d"].to_numpy()

print(f"Kích thước tập Train: {X_train.shape} (Nhãn dương: {y_train.sum()})")
print(f"Kích thước tập Val:   {X_val.shape} (Nhãn dương: {y_val.sum()})")
print(f"Kích thước tập Test:  {X_test.shape} (Nhãn dương: {y_test.sum()})")


## 14. Class Imbalance

Do tỷ lệ khách hàng rời đi rất nhỏ trong thực tế, tập dữ liệu gặp phải hiện tượng **mất cân bằng lớp cực đoan (Extreme Class Imbalance)**.


In [ ]:
# Tính toán tỷ lệ mất cân bằng lớp
total_rows = len(df)
churn_rows = int(df["churn_next_30d"].sum())
non_churn_rows = total_rows - churn_rows
churn_rate = churn_rows / total_rows

print(f"Tổng số bản ghi:       {total_rows}")
print(f"Khách hàng ở lại (0):  {non_churn_rows}")
print(f"Khách hàng rời bỏ (1): {churn_rows}")
print(f"Tỷ lệ Churn tổng thể:  {churn_rate:.4%}")


## 15. Baseline Models

Để có cơ sở so sánh, chúng ta xem lại kết quả huấn luyện của các mô hình baseline cơ sở:
- **Logistic Regression (Static)**:
  - PR-AUC Test: `0.053165`
  - F1-Score Test: `11.7647%`
- **Random Forest**:
  - PR-AUC Test: `0.103354`
  - F1-Score Test: `22.1130%`


## 16. Feature Reduction Experiment

Dự án đã chạy một thực nghiệm so sánh phương án **Rút gọn Đặc trưng thủ công (Selective Features)** nhằm giảm độ phức tạp:
- **Bộ Original (379 đặc trưng)**: PR-AUC Test = `0.104941`, F1-Score Test = `21.3531%`
- **Bộ Selective (76 đặc trưng thô)**: PR-AUC Test = `0.075032`, F1-Score Test = `14.7750%`

*Kết luận*: Việc rút gọn thủ công làm suy giảm đáng kể tín hiệu dự đoán rời bỏ dịch vụ nên phương án này bị loại bỏ.


## 17. Top-K Feature Selection Experiment

Đồng thời, thực nghiệm lựa chọn đặc trưng tự động dựa trên dữ liệu (Data-Driven Selection) cũng được benchmark để đo lường mức độ suy giảm hiệu năng đối với cấu hình sử dụng toàn bộ đặc trưng (All379):

| Feature Set | Feature Count | Test PR-AUC | Test F1-Score | PR-AUC Loss vs All379 |
| --- | --- | --- | --- | --- |
| **Top 50** | 50 | `0.100125` | `19.1910%` | `14.2886%` |
| **Top 100** | 100 | `0.109199` | `20.2381%` | `6.5207%` |
| **Top 150** | 150 | `0.113174` | `21.2638%` | `3.1179%` |
| **Top 200** | 200 | `0.111339` | `22.1402%` | `4.6887%` |
| **Top 250** | 250 | `0.111730` | `20.3759%` | `4.3544%` |
| **All 379** | 379 | `0.116816` | `22.0096%` | `0.0000%` |

*Kết luận*: Do các bộ đặc trưng thu nhỏ không đáp ứng đồng thời cả 2 tiêu chí điều kiện (PR-AUC loss $\le 3\%$ và F1 loss $\le 5\%$), dự án quyết định giữ lại toàn bộ 379 đặc trưng thời gian để tối đa hóa hiệu năng dự báo.


## 18. Final Feature Decision

Mô hình sản xuất chính thức sử dụng **Top 100 đặc trưng thời gian** (đã loại bỏ đa cộng tuyến $|r| > 0.98$ và lọc phương sai thấp strictly fit trên tập Train) để đảm bảo cân bằng tốt nhất giữa hiệu năng phân loại và tốc độ chạy.


## 19. Train Final LightGBM

Tải mô hình sản xuất đóng gói chính thức từ thư mục `artifacts/`.


In [ ]:
# Load final model bundle
artifact_path = Path("artifacts/temporal_churn_model.joblib")
bundle = joblib.load(artifact_path)

model = bundle["model"]
calibrator = bundle["calibrator"]
selected_features = bundle["selected_features"]
threshold = bundle["threshold"]
imputer = bundle["imputer"]

print(f"Đã tải thành công final model bundle từ: {artifact_path}")
print(f"Số đặc trưng đầu vào mô hình: {len(selected_features)}")
print(f"Ngưỡng quyết định tối ưu:     {threshold:.2f}")


## 20. Probability Calibration

Do phân bổ lệch xác suất thô của LightGBM sau khi phạt trọng số lớp dương (`scale_pos_weight`), mô hình sử dụng bộ hiệu chuẩn Platt Scaling (Logistic Regression) fit trên Validation để căn chỉnh lại xác suất dự báo sát với thực tế doanh nghiệp.


## 21. Threshold Selection

Ngưỡng quyết định mặc định của hệ thống đã được tối ưu hóa từ `0.50` thô dịch chuyển về ngưỡng calibrated **`0.07`** (hoặc `0.08` tùy snapshot) nhằm tối đa hóa F1-Score.


## 22. Final Evaluation

Thực hiện suy diễn và đánh giá mô hình trực tiếp trên tập kiểm thử độc lập Clean Test và vẽ biểu đồ trực quan.


In [ ]:
# Suy diễn trên tập Test
X_test_sel = test_df[selected_features].copy()
X_test_imp = imputer.transform(X_test_sel)

raw_probs = model.predict_proba(X_test_imp)[:, 1]
calibrated_probs = calibrator.predict_proba(raw_probs.reshape(-1, 1))[:, 1]
preds = (calibrated_probs >= threshold).astype(int)

# Tính toán các chỉ số kiểm thử
pr_auc = average_precision_score(y_test, calibrated_probs)
roc_auc = roc_auc_score(y_test, calibrated_probs)
precision = precision_score(y_test, preds, zero_division=0)
recall = recall_score(y_test, preds, zero_division=0)
f1 = f1_score(y_test, preds, zero_division=0)
cm = confusion_matrix(y_test, preds)

print("Kết quả đánh giá trên tập Clean Test sạch:")
print(f"  PR-AUC:    {pr_auc:.6f}")
print(f"  ROC-AUC:   {roc_auc:.6f}")
print(f"  Precision: {precision:.4%}")
print(f"  Recall:    {recall:.4%}")
print(f"  F1-Score:  {f1:.4%}")

# Vẽ Confusion Matrix
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix plot
ax[0].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax[0].set_title('Confusion Matrix (Test)')
ax[0].set_xticks([0, 1])
ax[0].set_yticks([0, 1])
ax[0].set_xticklabels(['Ở lại (0)', 'Rời đi (1)'])
ax[0].set_yticklabels(['Ở lại (0)', 'Rời đi (1)'])
ax[0].set_ylabel('Nhãn thực tế')
ax[0].set_xlabel('Nhãn dự báo')

# Thêm nhãn số lượng vào Confusion Matrix
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax[0].text(j, i, format(cm[i, j], 'd'),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black")

# Precision-Recall Curve
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, calibrated_probs)
ax[1].plot(recalls, precisions, color='darkorange', lw=2, label=f'PR Curve (AUC = {pr_auc:.4f})')
ax[1].set_xlabel('Recall (Độ phủ)')
ax[1].set_ylabel('Precision (Độ chính xác)')
ax[1].set_title('Precision-Recall Curve')
ax[1].set_xlim([0.0, 1.0])
ax[1].set_ylim([0.0, 1.05])
ax[1].legend(loc="lower left")
ax[1].grid(True)

plt.tight_layout()
plt.show()


### Vẽ Feature Importance

Vẽ biểu đồ Top 20 đặc trưng thời gian đóng đóng góp quan trọng nhất vào quyết định dự báo rời bỏ của mô hình LightGBM.


In [ ]:
# Lấy feature importance từ mô hình LightGBM
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=selected_features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.head(20).plot(kind='barh', color='teal').invert_yaxis()
plt.title('Top 20 Feature Importance (LightGBM)')
plt.xlabel('Điểm số quan trọng (Gain/Split)')
plt.ylabel('Tên đặc trưng')
plt.grid(axis='x', linestyle='--')
plt.show()


## 23. Save Model

Mô hình đã được huấn luyện, hiệu chuẩn và kiểm định hoàn thiện sẽ được lưu trữ đóng gói định kỳ vào tệp `artifacts/temporal_churn_model.joblib`.


## 24. Verify Saved Model

Kiểm định khả năng nạp lại và độ tin cậy của tệp đóng gói trên 100 dòng mẫu để tránh lỗi logic lúc triển khai suy diễn.


In [ ]:
# Nạp và kiểm định gói mô hình
loaded_bundle = joblib.load(artifact_path)
l_model = loaded_bundle["model"]
l_calibrator = loaded_bundle["calibrator"]
l_features = loaded_bundle["selected_features"]
l_threshold = loaded_bundle["threshold"]
l_imputer = loaded_bundle["imputer"]

# Chạy suy diễn thử trên 100 dòng mẫu
sample_df = df.head(100).copy()
X_sample = sample_df[l_features]

X_sample_imp = l_imputer.transform(X_sample)
s_raw = l_model.predict_proba(X_sample_imp)[:, 1]
s_cal = l_calibrator.predict_proba(s_raw.reshape(-1, 1))[:, 1]
s_preds = (s_cal >= l_threshold).astype(int)

# Chạy các xác thực
assert np.all(s_cal >= 0.0) and np.all(s_cal <= 1.0), "FAIL: Xác suất vượt quá giới hạn [0, 1]!"
assert np.all(np.isin(s_preds, [0, 1])), "FAIL: Dự báo phải là giá trị nhị phân {0, 1}!"

print("VERIFICATION COMPLETED: PASS")
print("Gói mô hình đóng gói hoạt động hoàn hảo và sẵn sàng phục vụ suy diễn.")


## 25. Kết luận

Dự án đã triển khai và kiểm duyệt thành công hệ thống dự báo Customer Churn sử dụng chuỗi hành vi thời gian hàng tháng trượt:

- **FINAL MODEL**: LightGBM
- **FEATURES**: Top 100 temporal features (All379 original candidate set)
- **TRAINING**: Rolling 12M Training Window
- **CALIBRATION**: Platt Scaling (Logistic Regression)
- **THRESHOLD**: 0.07 (Val F1: 23.1618%)
- **FINAL METRICS (Test)**: PR-AUC = 0.115641, ROC-AUC = 0.898023, F1 = 22.3816%
- **ARTIFACT**: `artifacts/temporal_churn_model.joblib`
- **PROJECT STATUS**: **COMPLETED**
